# 심근경색 합병증 예측 — EDA와 모델 학습 (Myocardial Infarction)

- 데이터: 심근경색 환자 1,700명 · 124열
- 목표: 합병증(예: 만성심부전 ZSN) 예측 — 이진분류
- 흐름: 불러오기 → 학습 전 확인 → EDA → 학습 → 해석
- 참고: 데이터 소개 data_MyocardialInfarction.txt

- 이 데이터는 **실전 난도 최상**
  (헤더 없음 · '?' 결측 · 시점 누수 · 결과열 12개 · 극단 결측)

## 1. 불러오기 — 헤더 없음, '?' 결측

- MI.data는 헤더가 없음 → header=None
- 결측이 '?'로 표기됨 → na_values="?" 없으면 결측을 못 잡음

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv("MI.data", header=None, na_values="?")
print(df.shape)          # (1700, 124)
print("결측 총합:", df.isna().sum().sum())   # na_values 덕에 잡힘
df.head()

### 1-1. 컬럼 위치 파악

- 컬럼명이 없으므로 **위치(인덱스)**로 다룬다
- col 0 = ID / col 1 = 나이 / col 2 = 성별
- col 112~123 = 결과 12개 (예측 대상 후보)
- 나머지(col 1~111) = 예측 변수

In [ ]:
# 컬럼에 이름 부여 (다루기 쉽게)
df = df.rename(columns={0:"ID", 1:"AGE", 2:"SEX", 120:"ZSN"})
# 결과 12개는 col 112~123
outcome_cols = list(range(112, 124))
print("결과 컬럼 위치:", outcome_cols)
print("타깃 ZSN(col120) 발생률:", (df["ZSN"]==1).mean().round(3))

## 2. 학습 전 확인 — 누수와 결측

- 이 데이터는 사람이 걸러낼 것이 많다

### 2-1. 다른 결과 11개는 예측 변수가 아님 (누수)

- 타깃 하나(ZSN)를 고르면, 나머지 결과 11개도 '미래 사건'
- 이들을 변수로 넣으면 다른 합병증 정보로 예측 = 누수
- ID도 제거 (식별자)

In [ ]:
# ID + 타깃 아닌 결과 11개 제거
drop_cols = ["ID"] + [c for c in outcome_cols if c != 120]
df_model = df.drop(columns=drop_cols)
print("제거 후:", df_model.shape)   # 타깃 ZSN은 남김

### 2-2. 입원 시점 예측이면 이후 시점 정보도 누수

- 일부 변수는 입원 후 24/48/72시간에 측정됨
- '입원 시점' 예측이라면 이후 값은 아직 없음 → 누수
- (심화) 시점 변수를 빼고 학습해 비교해 볼 수 있음
- 여기서는 기본 실습으로 전체 변수 사용, 심화는 문서 참조

### 2-3. 극단적 결측 컬럼

- 일부 컬럼은 결측이 95% 이상 (거의 값이 없음)
- AutoGluon이 처리는 하지만, 정보가 없어 제거가 나을 수 있음

In [ ]:
miss = df_model.isna().mean().sort_values(ascending=False)
print("결측률 상위 5:")
print(miss.head(5).round(3))

# 결측 90% 이상 컬럼 제거 (선택)
high_miss = miss[miss > 0.9].index
df_model = df_model.drop(columns=high_miss)
print("\n결측 과다 컬럼 제거 후:", df_model.shape)

## 3. EDA

- 변수가 많으므로(100+) minimal 모드로 가볍게

In [ ]:
from data_profiling import ProfileReport

profile = ProfileReport(df_model, minimal=True, progress_bar=False)
profile.to_file("mi_eda.html")   # 브라우저에서 열기
# minimal=True → 상관행렬 등 무거운 계산 생략

### 3-1. 타깃 분포

In [ ]:
import matplotlib.pyplot as plt
df_model["ZSN"].value_counts().plot(
    kind="bar", figsize=(5,3), title="ZSN (0=없음, 1=합병증)")
plt.show()
print("발생률:", (df_model["ZSN"]==1).mean().round(3))   # 약 23%

## 4. 학습

- 불균형(약 23%) → roc_auc로 평가
- 결측·인코딩은 내부 자동

In [ ]:
from autogluon.tabular import TabularPredictor
from sklearn.model_selection import train_test_split

train_data, test_data = train_test_split(
    df_model, test_size=0.2, random_state=42,
    stratify=df_model["ZSN"])

predictor = TabularPredictor(
    label="ZSN",
    eval_metric="roc_auc",
).fit(train_data, presets="medium_quality", time_limit=300)

In [ ]:
predictor.leaderboard(test_data)

## 5. 해석

In [ ]:
print(predictor.evaluate(test_data))

### 5-1. 변수 중요도

- 100개 넘는 변수 중 무엇이 합병증을 예측하는가
- 상위 변수가 의학적으로 말이 되는지 확인

In [ ]:
predictor.feature_importance(test_data).head(15)

## 정리

- 헤더 없음 → header=None · '?' → na_values 필수
- 결과 12개 중 타깃 1개만, 나머지 11개는 제거 (누수)
- 극단 결측 컬럼 제거 · 변수 많아 minimal EDA
- 불균형 → roc_auc
- 학습·앙상블은 TabularPredictor가 자동

- 이 데이터는 '실전이 이렇게 지저분하다'를 보여준다
  → 도구가 자동화해도, 무엇을 넣고 뺄지는 사람이 판단